# Expense RAG Prototype

A minimal RAG pipeline for the Personal Expense Intelligence Agent, using minsearch,
following the same structure used in the course: a `search` function, a `build_prompt`
function, an `llm` function, and a top-level `rag` function that chains them together.

This week's prototype uses a small synthetic dataset of Australian household expense
records (`data/expenses.json`) instead of real receipt images or OCR, so we can test
retrieval and RAG end to end before adding document extraction.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Synthetic household expense records for August and September 2026, covering
groceries, utilities, transport, dining, household, health, and subscriptions.
September includes a few deliberately unusual large expenses (a flight, a
replacement washing machine, a bulk grocery run, extra dining out) so the
"why were my expenses unusually high in September?" query has clear, traceable
evidence in the data.

Each record has: `id`, `date`, `merchant`, `category`, `description`, `total`.

In [2]:
with open("../data/expenses.json") as f:
    documents = json.load(f)

len(documents)

24

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

`merchant`, `category`, and `description` are indexed as free text, so a query
can match on the category name itself (e.g. "transport"), the merchant, or the
description. `category` is also indexed as a keyword field so it could be used
for exact-match filtering later (e.g. `filter_dict={"category": "household"}`).
`stop_words="english"` keeps common query words like "why" or "were" from
diluting the match on the words that actually matter, like "September".

In [3]:
index = Index(
    text_fields=["merchant", "category", "description"],
    keyword_fields=["category"],
    vectorizer_params={"stop_words": "english"},
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    return index.search(query, num_results=13)

In [5]:
instructions = """
You're a personal expense assistant. Answer the QUESTION using only the
expense records provided in the CONTEXT below - each record is a household
transaction with a merchant, category, description, date, and total.

When you answer:
- Identify the specific merchants, categories, dates, and amounts that support
  your answer.
- If the question asks "why" something changed (e.g. a spending increase),
  point to the specific transactions that explain it.
- If the CONTEXT doesn't contain enough information to answer confidently,
  say so clearly instead of guessing.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

Three representative queries against the synthetic expense data.

In [8]:
queries = [
    "Why were my expenses unusually high in September?",
    "What were my largest household expenses?",
    "Which transactions were related to transport?",
]

for q in queries:
    print(f"Q: {q}")
    print(rag(q))
    print("-" * 80)

Q: Why were my expenses unusually high in September?


Your expenses were unusually high in September due to several significant transactions across different categories:

1. **Utilities**:
   - **Origin Energy**: September electricity bill - **$122.50** (September 6)
   - **Sydney Water**: September water bill - **$41.30** (September 8)
   - **Telstra**: September mobile and home internet plan - **$95.00** (September 1)

2. **Dining**:
   - **The Local Hotel**: September birthday dinner for two - **$92.50** (September 5)
   - **Guzman y Gomez**: September dinner with friends - **$48.70** (September 12)

3. **Transport**:
   - **Qantas**: Flight from Sydney to Melbourne - **$389.00** (September 11)
   - **Uber**: Rides to and from the airport - **$67.40** (September 20)

4. **Household**:
   - **Bunnings Warehouse**: Replacement washing machine - **$648.00** (September 14)
   - **Kmart**: New desk and chair for the home office - **$149.00** (September 21)

5. **Groceries**:
   - **Woolworths**: Grocery shopping - **$101.20** (September 4)


Your largest household expenses are as follows:

1. **Bunnings Warehouse**
   - **Category**: Household
   - **Date**: September 14, 2026
   - **Amount**: $648.00
   - **Description**: Replacement washing machine bought after the old one broke.

2. **Kmart**
   - **Category**: Household
   - **Date**: September 21, 2026
   - **Amount**: $149.00
   - **Description**: New desk and chair for the home office.

3. **Kmart**
   - **Category**: Household
   - **Date**: August 15, 2026
   - **Amount**: $54.00
   - **Description**: New cushions and storage boxes for the living room.

These transactions reflect your largest expenses in the household category. The replacement washing machine from Bunnings Warehouse is the most significant one at $648.00.
--------------------------------------------------------------------------------
Q: Which transactions were related to transport?


The transactions related to transport are:

1. **Uber**
   - **Date:** 2026-08-10
   - **Description:** August commute rides to work
   - **Total:** $46.75

2. **Ampol**
   - **Date:** 2026-08-20
   - **Description:** August fuel fill-up for the car
   - **Total:** $58.40

3. **Qantas**
   - **Date:** 2026-09-11
   - **Description:** September flight from Sydney to Melbourne for a family visit
   - **Total:** $389.00

4. **Uber**
   - **Date:** 2026-09-20
   - **Description:** September rides to and from the airport
   - **Total:** $67.40
--------------------------------------------------------------------------------
